In [ ]:
# Example 1: Build a basic Activity (the core protocol message type).
from microsoft_agents.activity import Activity, ActivityTypes

activity = Activity(
    type=ActivityTypes.message,
    id="msg-001",
    text="hello from the smoke test",
    locale="en-US",
)

print(f"type={activity.type!r}")
print(f"id={activity.id!r}")
print(f"text={activity.text!r}")

assert activity.type == ActivityTypes.message
assert activity.text == "hello from the smoke test"

In [ ]:
# Example 2: ChannelAccount + ConversationAccount round-trip through pydantic.
from microsoft_agents.activity import ChannelAccount, ConversationAccount

sender = ChannelAccount(id="user-42", name="Ada Lovelace", role="user")
convo = ConversationAccount(id="conv-7", name="smoke-test-room", is_group=False)

sender_dump = sender.model_dump()
convo_dump = convo.model_dump()

print(f"sender: {sender_dump}")
print(f"convo:  {convo_dump}")

assert sender.id == "user-42"
assert convo.id == "conv-7"
assert convo.is_group is False

In [ ]:
# Example 3: HeroCard + Attachment + CardAction (the card composition surface).
from microsoft_agents.activity import (
    Attachment,
    CardAction,
    HeroCard,
)

card = HeroCard(
    title="Greetings",
    subtitle="From the test suite",
    text="This is a card body.",
    buttons=[CardAction(type="openUrl", title="open", value="https://example.org")],
)
attachment = Attachment(
    content_type="application/vnd.microsoft.card.hero",
    content=card.model_dump(),
)

print(f"card title: {card.title}")
print(f"button[0] type: {card.buttons[0].type}")
print(f"attachment content_type: {attachment.content_type}")

assert attachment.content_type.endswith("hero")
assert len(card.buttons) == 1

In [ ]:
# Example 4: JSON round-trip — serialize an Activity and parse it back.
from microsoft_agents.activity import Activity, ActivityTypes, ChannelAccount

original = Activity(
    type=ActivityTypes.message,
    id="msg-rt-1",
    text="round trip",
    from_property=ChannelAccount(id="u1", name="Alice"),
)

# Serialize with exclude_unset — Activity fields are non-Optional-typed with None
# defaults, so explicit nulls fail validation on the way back in (the SDK's wire
# format also omits unset fields).
as_json = original.model_dump_json(exclude_unset=True)
print(f"serialized length: {len(as_json)} chars")

restored = Activity.model_validate_json(as_json)
print(f"restored id={restored.id} text={restored.text!r}")
print(f"restored.from_property.name={restored.from_property.name}")

assert restored.id == original.id
assert restored.text == original.text
assert restored.from_property.name == "Alice"

In [ ]:
# Example 5: Enums + Entity types — exercise the constant tables and Mention entity.
from microsoft_agents.activity import (
    ActivityTypes,
    Channels,
    Mention,
    ChannelAccount,
    RoleTypes,
    InputHints,
)

print(f"ActivityTypes.message = {ActivityTypes.message}")
print(f"Channels.msteams      = {Channels.ms_teams}")
print(f"RoleTypes.user        = {RoleTypes.user}")
print(f"InputHints.accepting  = {InputHints.accepting_input}")

mention = Mention(
    type="mention",
    mentioned=ChannelAccount(id="u-7", name="Bob"),
    text="@Bob",
)
print(f"mention: type={mention.type} text={mention.text!r}")

assert str(ActivityTypes.message) == "message" or ActivityTypes.message.value == "message"
assert mention.mentioned.id == "u-7"